# March Madness 2025 Prediction Model

## Goal and Setup
This notebook aims to build a complete prediction model for the 2025 NCAA basketball tournaments.

We will load and process data, train machine learning models, and generate predictions for tournament games.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Define the data path
data_path = "data/"

## Loading Data

In [3]:
# Load all necessary datasets including team information, game results, and tournament seeds
cities = pd.read_csv(os.path.join(data_path, "Cities.csv"))
conferences = pd.read_csv(os.path.join(data_path, "Conferences.csv"))
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))
mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))
mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))
mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

## Defining Data Processing Functions

In [4]:
# Extract numeric seed values
def extract_seed(seed):
    return int(seed[1:3])

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

In [5]:
# Compute team statistics
def compute_team_stats(results):
    team_stats = results.groupby("WTeamID").agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum"
    })
    team_stats.columns = ["_".join(col).strip() for col in team_stats.columns.values]
    return team_stats.reset_index()

mteam_stats = compute_team_stats(mregular_results)
wteam_stats = compute_team_stats(wregular_results)

## Preparing Training Data

In [6]:
def prepare_training_data(results, seeds):
    # Merge in seed information for both winning and losing teams
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left")
    results = results.rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left")
    results = results.rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    
    # Fill missing values
    results["WSeed"].fillna(results["WSeed"].median(), inplace=True)
    results["LSeed"].fillna(results["LSeed"].median(), inplace=True)
    
    # Create feature set for wins
    win_features = results[["WTeamID", "LTeamID", "WSeed", "LSeed"]].copy()
    win_features["SeedDiff"] = win_features["WSeed"] - win_features["LSeed"]
    win_features["Win"] = 1  # Label indicating the winning team
    
    # Create feature set for losses (swapping WTeamID and LTeamID)
    loss_features = results[["LTeamID", "WTeamID", "LSeed", "WSeed"]].copy()
    loss_features.columns = ["WTeamID", "LTeamID", "WSeed", "LSeed"]
    loss_features["SeedDiff"] = loss_features["WSeed"] - loss_features["LSeed"]
    loss_features["Win"] = 0  # Label indicating the losing team
    
    # Combine both datasets
    full_features = pd.concat([win_features, loss_features], ignore_index=True)
    
    return full_features

In [7]:
# Prepare training data for men's and women's tournaments
mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds)

In [8]:
# Combine men's and women's data
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

In [9]:
print(full_train_data["Win"].value_counts())

Win
1    4168
0    4168
Name: count, dtype: int64


## Splitting and Scaling Data

In [10]:
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

## Training Multiple Models

In [11]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier()
}

best_model = None
best_score = float("inf")

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict_proba(X_valid_scaled)[:, 1]
    score = log_loss(y_valid, y_pred)
    print(f"{name} Log Loss: {score}")
    if score < best_score:
        best_score = score
        best_model = model

Logistic Regression Log Loss: 0.5144231758839312
Random Forest Log Loss: 0.7736244561082148
Gradient Boosting Log Loss: 0.5039794230921193
XGBoost Log Loss: 0.5544494087448302


## Training Final Model

In [12]:
best_model.fit(X_train_scaled, y_train)

GradientBoostingClassifier()

## Generating Tournament Predictions

In [13]:
# Load the correct sample submission file
submission_df = pd.read_csv(os.path.join(data_path, "SampleSubmissionStage2.csv"))

In [14]:
# Split ID into Season, WTeamID, LTeamID
submission_df[["Season", "WTeamID", "LTeamID"]] = submission_df["ID"].str.split("_", expand=True).astype(int)

In [15]:
# Merge seed values for both teams
submission_df = submission_df.merge(mtourney_seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
submission_df = submission_df.merge(mtourney_seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])


In [16]:
# Check for missing seeds BEFORE filling
print("Missing WSeed BEFORE filling:", submission_df["WSeed"].isna().sum())
print("Missing LSeed BEFORE filling:", submission_df["LSeed"].isna().sum())


Missing WSeed BEFORE filling: 131407
Missing LSeed BEFORE filling: 131407


In [17]:
# Fill missing values with the median seed
median_seed = mtourney_seeds["Seed"].median()
submission_df["WSeed"].fillna(median_seed, inplace=True)
submission_df["LSeed"].fillna(median_seed, inplace=True)

In [18]:
# Check for missing values AFTER filling
print("Missing WSeed AFTER filling:", submission_df["WSeed"].isna().sum())
print("Missing LSeed AFTER filling:", submission_df["LSeed"].isna().sum())

Missing WSeed AFTER filling: 0
Missing LSeed AFTER filling: 0


In [19]:
# Compute feature SeedDiff
submission_df["SeedDiff"] = submission_df["WSeed"] - submission_df["LSeed"]

In [20]:
# Compute feature SeedDiff
submission_df["SeedDiff"] = submission_df["WSeed"] - submission_df["LSeed"]

In [21]:
# Prepare feature set for prediction
submission_features = submission_df[["WTeamID", "LTeamID", "WSeed", "LSeed", "SeedDiff"]]

In [22]:
# Debug: Ensure no missing values before scaling
print("Missing values in submission_features:", submission_features.isna().sum().sum())

Missing values in submission_features: 0


In [23]:
# Scale features
submission_features_scaled = scaler.transform(submission_features)

In [24]:
# Double-check data shape
print(f"Shape of submission_features_scaled: {submission_features_scaled.shape}")

Shape of submission_features_scaled: (131407, 5)


In [25]:
# Generate predictions
submission_df["Pred"] = best_model.predict_proba(submission_features_scaled)[:, 1]

In [26]:
# Save submission file
submission_df[["ID", "Pred"]].to_csv("submission.csv", index=False)

In [ ]:
# Final check on submission shape
print(f"Submission file shape: {submission_df[['ID', 'Pred']].shape}")

Submission file shape: (131407, 2)


: 